# MAT 504 — Computer Modelling in Materials Science
## Week 3 Hands-on — From Atomic Trajectories to Diffusion

### Objective

In Week 3, we connected a microscopic model to a macroscopic transport property. In this hands-on exercise, you will investigate how the **molecular-dynamics time step** affects numerical reliability and then use an atomistic trajectory to calculate a diffusion coefficient.

The conceptual sequence is:

**potential-energy model → forces → Verlet integration → atomic trajectories → mean-squared displacement → diffusion coefficient → continuum concentration profile**.

We use a small two-dimensional Lennard–Jones fluid in reduced units,

\[
\epsilon=\sigma=m=k_B=1.
\]

### Working with the notebook

Complete the indicated `TODO` items and short written questions as you work through the exercise. Your notebook should run from top to bottom without manual intervention.

Read the supplied helper functions as well as the code you complete yourself so that you can connect each computational step to the molecular-dynamics concepts from the lecture.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = False

# Reproducibility
SEED = 504


## Model and supplied helper functions

The Lennard–Jones pair potential is

\[
U(r)=4\left(r^{-12}-r^{-6}\right).
\]

The simulation uses a square periodic box and the minimum-image convention. The trajectory returned by the simulation is **unwrapped**, so a particle that crosses a periodic boundary continues smoothly in the stored trajectory.

The position-Verlet update is

\[
\mathbf r_i(t+\Delta t)
=
2\mathbf r_i(t)-\mathbf r_i(t-\Delta t)
+
\mathbf a_i(t)\Delta t^2.
\]

Read the functions below carefully. You do not need to rewrite them, but you should be able to explain what they do.


In [ ]:
def initialize_lattice(n_particles, box_length):
    """Place particles on a square lattice to avoid initial overlaps."""
    n_side = int(np.ceil(np.sqrt(n_particles)))
    spacing = box_length / n_side
    coords = np.arange(n_side) * spacing + 0.5 * spacing
    positions = np.array([(x, y) for x in coords for y in coords], dtype=float)
    return positions[:n_particles]


def lj_forces_and_energy(positions_wrapped, box_length):
    """Return Lennard-Jones forces and potential energy in reduced units."""
    r = positions_wrapped

    # Pair displacement r_i - r_j
    dr = r[:, None, :] - r[None, :, :]

    # Minimum-image convention for periodic boundaries
    dr -= box_length * np.rint(dr / box_length)

    r2 = np.sum(dr**2, axis=2)
    np.fill_diagonal(r2, np.inf)

    inv_r2 = 1.0 / r2
    inv_r6 = inv_r2**3

    # F_ij = 24 [2 r^-14 - r^-8] r_ij
    prefactor = 24.0 * inv_r2 * (2.0 * inv_r6**2 - inv_r6)
    prefactor[np.isinf(r2)] = 0.0
    forces = np.sum(prefactor[:, :, None] * dr, axis=1)

    # Pair potential; factor 1/2 removes double counting
    pair_u = 4.0 * (inv_r6**2 - inv_r6)
    np.fill_diagonal(pair_u, 0.0)
    potential_energy = 0.5 * np.sum(pair_u)

    return forces, potential_energy


def run_position_verlet(
    dt,
    total_time=30.0,
    n_particles=36,
    density=0.65,
    initial_temperature=1.0,
    sample_dt=0.05,
    seed=504,
):
    """Run a 2D NVE Lennard-Jones simulation using position Verlet."""
    box_length = np.sqrt(n_particles / density)
    positions0 = initialize_lattice(n_particles, box_length)

    rng = np.random.default_rng(seed)

    # Initial velocities; remove center-of-mass motion
    velocities0 = rng.normal(size=(n_particles, 2))
    velocities0 -= velocities0.mean(axis=0)

    # Scale the initial kinetic temperature to the requested value
    dof = 2 * n_particles - 2
    current_T = np.sum(velocities0**2) / dof
    velocities0 *= np.sqrt(initial_temperature / current_T)

    # Initial acceleration
    forces0, _ = lj_forces_and_energy(positions0, box_length)
    acceleration0 = forces0  # m = 1 in reduced units

    # Position at t = -dt, obtained from a Taylor expansion
    r_current = positions0.copy()          # unwrapped coordinates
    r_previous = (
        r_current
        - velocities0 * dt
        + 0.5 * acceleration0 * dt**2
    )

    n_steps = int(round(total_time / dt))
    sample_every = max(1, int(round(sample_dt / dt)))

    times = []
    trajectory = []
    total_energy = []

    for step in range(n_steps):
        # Wrap only for force evaluation
        r_wrapped = np.mod(r_current, box_length)
        forces, potential_energy = lj_forces_and_energy(r_wrapped, box_length)
        acceleration = forces  # m = 1

        # Position-Verlet / central-difference update
        r_new = 2.0 * r_current - r_previous + acceleration * dt**2

        # Central-difference velocity at the current time
        velocity = (r_new - r_previous) / (2.0 * dt)
        kinetic_energy = 0.5 * np.sum(velocity**2)

        if step % sample_every == 0:
            times.append(step * dt)
            trajectory.append(r_current.copy())
            total_energy.append(kinetic_energy + potential_energy)

        r_previous, r_current = r_current, r_new

    return {
        "time": np.asarray(times),
        "trajectory": np.asarray(trajectory),
        "energy": np.asarray(total_energy),
        "box_length": box_length,
        "dt": dt,
    }


## Part 1 — Predict the effect of the time step

We compare

\[
\Delta t_{\mathrm{ref}}=0.005,
\qquad
\Delta t_{\mathrm{large}}=0.025.
\]

Both simulations start from the same configuration and velocities and run for the same total physical time.

### Question 1

**a.** Which time step do you expect to conserve total energy more accurately in an NVE trajectory?

**b.** Why can increasing \(\Delta t\) reduce the accuracy of Verlet integration even though the underlying equations of motion have not changed?

**Answer:**

_Write your answer here._


## Part 2 — Time-step sensitivity and energy conservation

Run the two simulations. The larger time step is **intentionally poor**; do not change it.


In [ ]:
dt_ref = 0.005
dt_large = 0.025

# TODO 1: run the reference-time-step simulation
run_ref = ...

# TODO 2: run the large-time-step simulation
run_large = ...


### Task 2.1 — Relative energy change

For each trajectory calculate

\[
\Delta E_{\mathrm{rel}}(t)
=
100\times\frac{E(t)-E(0)}{|E(0)|}.
\]

Plot both curves on the same axes.


In [ ]:
def relative_energy_change_percent(energy):
    # TODO: return 100 * (E(t) - E(0)) / |E(0)|
    return ...

# TODO: calculate the two relative-energy-change arrays
energy_change_ref = ...
energy_change_large = ...

plt.figure(figsize=(8, 4.5))
# TODO: plot both curves against their corresponding time arrays

plt.axhline(0, linewidth=0.8)
plt.xlabel("Time (reduced units)")
plt.ylabel("Relative energy change (%)")
plt.title("Time-step sensitivity of total energy")
plt.legend()
plt.tight_layout()
plt.show()


### Task 2.2 — Quantitative comparison

For each run report:

- the final relative energy change;
- the maximum absolute relative energy deviation.


In [ ]:
# TODO: calculate these four quantities
final_ref = ...
max_abs_ref = ...
final_large = ...
max_abs_large = ...

print(f"Reference dt = {dt_ref:.3f}")
print(f"  Final relative energy change: {final_ref:.3f} %")
print(f"  Maximum |relative change|:    {max_abs_ref:.3f} %")

print(f"\nLarge dt = {dt_large:.3f}")
print(f"  Final relative energy change: {final_large:.3f} %")
print(f"  Maximum |relative change|:    {max_abs_large:.3f} %")


### Question 2

**a.** Which trajectory is numerically more trustworthy?

**b.** Does the poor trajectory necessarily look obviously wrong if you inspect only the particle coordinates?

**c.** Why is energy conservation a useful diagnostic for an NVE molecular-dynamics calculation?

**Answer:**

_Write your answer here._


## Part 3 — From atomic trajectories to the diffusion coefficient

Use the **reference-time-step trajectory** for the main analysis in this section.

### Task 3.1 — Follow one particle

Plot the unwrapped \(x\)-\(y\) trajectory of particle 0. Mark its starting and final positions.


In [ ]:
trajectory_ref = run_ref["trajectory"]
particle_index = 0

# TODO: extract the unwrapped trajectory of particle 0
particle_path = ...

plt.figure(figsize=(5.5, 5.5))
# TODO: plot x versus y for particle_path
# TODO: mark the initial point and the final point with distinct markers

plt.xlabel("x (unwrapped)")
plt.ylabel("y (unwrapped)")
plt.title("Trajectory of particle 0")
plt.axis("equal")
plt.legend()
plt.tight_layout()
plt.show()


### Question 3

Why is this path qualitatively consistent with the microscopic picture of diffusion discussed in the lecture?

**Answer:**

_Write your answer here._


### Task 3.2 — Ensemble mean-squared displacement

Calculate

\[
\mathrm{MSD}(t)
=
\frac{1}{N}
\sum_{i=1}^{N}
\left|\mathbf r_i(t)-\mathbf r_i(0)\right|^2.
\]

Because the trajectory is unwrapped, boundary crossings do not artificially shorten the displacement.


In [ ]:
def mean_squared_displacement(trajectory):
    # trajectory shape: (n_frames, n_particles, 2)

    # TODO 1: displacement of every particle relative to its initial position
    displacement = ...

    # TODO 2: squared magnitude of each displacement vector
    squared_displacement = ...

    # TODO 3: average over particles at each time
    msd = ...

    return msd

# TODO: calculate the reference MSD
msd_ref = ...

plt.figure(figsize=(7.5, 4.5))
# TODO: plot reference MSD versus time
plt.xlabel("Time (reduced units)")
plt.ylabel("MSD")
plt.title("Mean-squared displacement — reference trajectory")
plt.tight_layout()
plt.show()


### Question 4

Why is averaging over many particles more useful than using the displacement of a single particle when estimating a transport property such as \(D\)?

**Answer:**

_Write your answer here._


### Task 3.3 — Obtain the diffusion coefficient

For normal diffusion in \(d\) dimensions,

\[
\mathrm{MSD}(t)=2dDt+C.
\]

For this two-dimensional simulation,

\[
D=\frac{1}{4}\frac{d\,\mathrm{MSD}}{dt}.
\]

Use the fixed fitting interval

\[
5\le t\le25.
\]


In [ ]:
def fit_diffusion_coefficient(time, msd, fit_start=5.0, fit_end=25.0, dimensions=2):
    mask = (time >= fit_start) & (time <= fit_end)

    # TODO 1: linear fit MSD = slope * t + intercept
    slope, intercept = ...

    # TODO 2: convert the slope to D using slope = 2 d D
    D = ...

    return D, slope, intercept, mask

# TODO: fit the reference trajectory
D_ref, slope_ref, intercept_ref, fit_mask_ref = ...

print(f"Reference-run slope = {slope_ref:.5f}")
print(f"Reference-run D     = {D_ref:.5f}")

plt.figure(figsize=(7.5, 4.5))
plt.plot(run_ref["time"], msd_ref, label="MSD")

# TODO: overlay the fitted straight line over the fitting interval

plt.xlabel("Time (reduced units)")
plt.ylabel("MSD")
plt.title("Diffusive fit — reference trajectory")
plt.legend()
plt.tight_layout()
plt.show()


### Task 3.4 — What if the time step is poor?

Calculate the MSD and apparent diffusion coefficient for the large-time-step trajectory using the same fitting interval.


In [ ]:
# TODO 1: calculate the large-time-step MSD
msd_large = ...

# TODO 2: fit the apparent diffusion coefficient
D_large, slope_large, intercept_large, fit_mask_large = ...

print(f"Reference dt: D = {D_ref:.5f}")
print(f"Large dt:     D = {D_large:.5f}")

plt.figure(figsize=(7.5, 4.5))
# TODO: plot both MSD curves for comparison
plt.xlabel("Time (reduced units)")
plt.ylabel("MSD")
plt.title("MSD comparison")
plt.legend()
plt.tight_layout()
plt.show()


### Question 5

Can a smooth-looking MSD curve or a plausible value of \(D\) compensate for poor numerical energy conservation? Why or why not?

**Answer:**

_Write your answer here._


## Part 4 — Return to the continuum description

Use the diffusion coefficient from the **reference trajectory** in

\[
c(x,t)
=
\frac{1}{\sqrt{4\pi Dt}}
\exp\left(-\frac{x^2}{4Dt}\right).
\]

Plot the concentration profile at \(t=2\) and \(t=10\).


In [ ]:
def gaussian_diffusion_profile(x, t, D):
    # TODO: implement the 1D continuum solution
    return ...

x = np.linspace(-5.0, 5.0, 1000)
times_to_plot = [2.0, 10.0]

plt.figure(figsize=(7.5, 4.5))
for t in times_to_plot:
    # TODO: calculate and plot c(x,t) using D_ref
    pass

plt.xlabel("x")
plt.ylabel("c(x,t)")
plt.title("Continuum diffusion profiles using MD-derived D")
plt.legend()
plt.tight_layout()
plt.show()


### Question 6

**a.** How does the concentration profile change with time?

**b.** What information from the atomistic simulation is retained in the continuum description through \(D\)?

**c.** What microscopic information is no longer visible once the system is represented only through \(c(x,t)\) and \(D\)?

**Answer:**

_Write your answer here._


## Final synthesis

In **3–4 sentences**, explain the complete connection

\[
U(\mathbf r)
\rightarrow
\mathbf F
\rightarrow
\text{Verlet trajectory}
\rightarrow
\mathrm{MSD}(t)
\rightarrow
D
\rightarrow
c(x,t).
\]

Make clear which steps are microscopic/atomistic and which step represents the transition to a continuum description.

**Answer:**

_Write your answer here._


## Before you leave

Make sure your notebook contains:

1. your prediction about time-step accuracy;
2. the two relative-energy-change curves;
3. the numerical energy-conservation comparison;
4. the trajectory of one particle;
5. the reference-run MSD and fitted diffusive region;
6. the diffusion coefficients from the reference and large-time-step runs;
7. the continuum concentration profiles generated using the reference \(D\);
8. all written answers and the final synthesis.
